In [51]:
#Day 4, Topic 1: The Split‑Apply‑Combine Paradigm

import pandas as pd
df = pd.read_csv('titanic.csv')

In [52]:
#The Simplest GroupBy – One Column, One Aggregation
# Split by 'Pclass', apply .mean() to 'Age', combine

avg_age_by_class = df.groupby('Pclass')['Age'].mean().astype(int)
avg_age_by_class

Pclass
1    38
2    29
3    25
Name: Age, dtype: int64

In [53]:
#Visualizing the Split

grouped = df.groupby('Pclass')
print(grouped.groups.keys())
grouped.get_group(1).head(2)

dict_keys([1, 2, 3])


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


In [54]:
#Multiple Aggregations on a Single Column
age_stats = df.groupby('Pclass')['Age'].agg(['mean', 'median', 'min', 'max', 'std'])
age_stats

,mean,median,min,max,std
Pclass,,,,,
1,38.233441,37.0,0.92,80.0,14.802856
2,29.877630,29.0,0.67,70.0,14.001077
3,25.140620,24.0,0.42,74.0,12.495398


In [55]:
#Grouping by Multiple Columns
# Survival rate by class and sex

survival_by_class_sex = df.groupby(['Pclass', 'Sex'])['Survived'].mean()
survival_by_class_sex

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

In [56]:
#Grouping with .size() and .count()
print(df.groupby('Pclass').size())

# How many non‑missing ages per class?
print(df.groupby('Pclass')['Age'].count())

Pclass
1    216
2    184
3    491
dtype: int64
Pclass
1    186
2    173
3    355
Name: Age, dtype: int64


In [57]:
#The .agg() Method – Different Functions for Different Columns
result = df.groupby('Pclass').agg({
    'Age': 'mean',
    'Fare': ['max', 'min'],
    'Survived': 'sum'
})
print(result)

              Age      Fare      Survived
             mean       max  min      sum
Pclass                                   
1       38.233441  512.3292  0.0      136
2       29.877630   73.5000  0.0       87
3       25.140620   69.5500  0.0      119


In [58]:
#Grouping by a Derived Column
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 18, 35, 60, 100], labels=['Child', 'Young Adult', 'Adult', 'Senior'])

df.groupby('AgeGroup', observed=False)['Survived'].mean()

AgeGroup
Child          0.503597
Young Adult    0.382682
Adult          0.400000
Senior         0.227273
Name: Survived, dtype: float64

In [59]:
#Practice Activity: Basic GroupBy

import pandas as pd
df = pd.read_csv('titanic.csv')

In [60]:
#Task 1: Use .groupby() to calculate the average fare for each embarkation point (Embarked). 
# Which port had the highest average fare?

avg_fare = df.groupby('Embarked')['Fare'].mean()

avg_fare.idxmax()

'C'

In [61]:
"""Task 2: Use .groupby() to find the maximum, minimum, and mean age for each passenger class (Pclass). 
Use a single .agg() call."""

age_stats = df.groupby('Pclass')['Age'].agg(['max', 'min', 'mean'])
age_stats

,max,min,mean
Pclass,,,
1,80.0,0.92,38.233441
2,70.0,0.67,29.877630
3,74.0,0.42,25.140620


In [62]:
#Task 3: Group by both Sex and Pclass, and calculate the count of passengers in each group. Use .size().

passen_stats = df.groupby(['Sex', 'Pclass']).size()
passen_stats

Sex     Pclass
female  1          94
        2          76
        3         144
male    1         122
        2         108
        3         347
dtype: int64

In [63]:
"""Task 4: Create a new column FamilySize = SibSp + Parch + 1. Then group by FamilySize and 
compute the survival rate (mean of Survived). Which family size had the highest survival rate?"""

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

family_sur_stats =df.groupby('FamilySize')['Survived'].mean()


family_sur_stats.idxmax()

np.int64(4)

In [64]:
"""Task 5 (Bonus): Group by the first letter of the passenger's name (use .str[0]). 
How many passengers have names starting with each letter? Which letter is most common?"""

lett_counts = df.groupby(df['Name'].str[0]).size()

lett_counts.idxmax()

'S'

In [65]:
#Day 4, Topic 2: Basic Aggregations with .groupby()

import pandas as pd

df = pd.read_csv('titanic.csv')

In [66]:
#Using .agg() with a List of Functions

age_stats = df.groupby('Sex')['Age'].agg(['mean', 'std', 'count', 'nunique'])

age_stats

,mean,std,count,nunique
Sex,,,,
female,27.915709,14.110146,261,63
male,30.726645,14.678201,453,82


In [67]:
#Different Aggregations for Different Columns
result = df.groupby('Pclass').agg({
    'Age': 'median', 
    'Fare': ['mean', 'max'],
    'Survived': 'sum'
})

result

Age       Fare           Survived
       median       mean       max      sum
Pclass                                     
1        37.0  84.154687  512.3292      136
2        29.0  20.662183   73.5000       87
3        24.0  13.675550   69.5500      119

In [68]:
#Renaming Aggregated Columns
result = df.groupby('Embarked').agg(
    avg_fare=('Fare', 'mean'),
    total_passenger=('PassengerId', 'count'),
    survived_count=('Survived', 'sum')
)

result

,avg_fare,total_passenger,survived_count
Embarked,,,
C,59.954144,168,93
Q,13.276030,77,30
S,27.079812,644,217


In [69]:
#Aggregating Without Grouping – .agg() on Entire DataFrame
df[['Age', 'Fare']].agg(['mean', 'median', 'std'])

,Age,Fare
mean,29.699118,32.204208
median,28.000000,14.454200
std,14.526497,49.693429


In [70]:
#Practice Activity: Aggregations

import pandas as pd

df = pd.read_csv('titanic.csv')

In [71]:
#Task 1: For each Pclass, compute the sum, mean, and max of the Fare column using a single .agg() call with a list.

fare_stats = df.groupby('Pclass')['Fare'].agg(['sum', 'mean', 'max'])
fare_stats

,sum,mean,max
Pclass,,,
1,18177.4125,84.154687,512.3292
2,3801.8417,20.662183,73.5000
3,6714.6951,13.675550,69.5500


In [72]:
"""Task 2: For each Embarked port, compute:
Average Age
Total Fare collected
Number of survivors (Survived sum)
Use a dictionary inside .agg()."""

embarked_stats = df.groupby('Embarked').agg({
    'Age': 'mean', 
    'Fare': 'sum',
    'Survived': 'sum'
})
embarked_stats

,Age,Fare,Survived
Embarked,,,
C,30.814769,10072.2962,93
Q,28.089286,1022.2543,30
S,29.445397,17439.3988,217


In [73]:
"""Task 3: Use named aggregations to produce a DataFrame with columns:
avg_age
max_fare
survival_rate (mean of Survived)
Group by Sex."""

sex_stats = df.groupby('Sex').agg(
    avg_age=('Age', 'mean'),
    max_fare=('Fare', 'max'),
    survival_rate=('Survived', 'mean')
)

sex_stats

,avg_age,max_fare,survival_rate
Sex,,,
female,27.915709,512.3292,0.742038
male,30.726645,512.3292,0.188908


In [74]:
"""Task 4: Compute the median and interquartile range (IQR) of Fare for each Pclass.
IQR = 75th percentile – 25th percentile. Use .quantile() inside .agg()."""

def get_iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

fare_quan_avg = df.groupby('Pclass')['Fare'].agg(['median', get_iqr])

fare_quan_avg

,median,get_iqr
Pclass,,
1,60.2875,62.57605
2,14.2500,13.00000
3,8.0500,7.75000


In [75]:
"""Task 5 (Bonus): Group by Pclass and Sex, then compute the count, mean, and std of Age. 
Use a single .agg() call with a list for Age only."""

age_stats = df.groupby(['Pclass', 'Sex'])['Age'].agg(['count', 'mean', 'std'])

age_stats

count       mean        std
Pclass Sex                                
1      female     85  34.611765  13.612052
       male      101  41.281386  15.139570
2      female     74  28.722973  12.872702
       male       99  30.740707  14.793894
3      female    102  21.750000  12.729964
       male      253  26.507589  12.159514